# Notebook 03: Đánh giá Đa chiều ASR, MT Cascade & Kiểm định Thống kê Bootstrap
**Dự án:** Vietnamese ASR → English Translation trên Kaggle T4 ×2

### Mục tiêu:
1. Đánh giá độ chính xác nhận dạng ASR (WER/CER toàn cục và phân tích theo 3 miền Bắc - Trung - Nam).
2. Đánh giá Baseline ngoại vi: Whisper-small zero-shot multilingual trên cùng tập test ViMD.
3. Đánh giá chất lượng dịch thuật Cascade: ASR transcriptions → VinAI Translate vi2en-v2 (sacreBLEU, chrF++).
4. Kiểm định thống kê Paired Bootstrap (1,000 resamples): 95% Confidence Intervals và p-value cho B1 vs P1, P1 vs P2.
5. Xuất báo cáo khoa học so sánh toàn diện.

In [ ]:
# Kiểm tra mã nguồn dự án và thiết bị
%cd /kaggle/working
from pathlib import Path
import torch

work_dir = Path("/kaggle/working/vi_asr_project")
if not (work_dir / "pyproject.toml").exists():
    !rm -rf /kaggle/working/vi_asr_project
    !git clone --recurse-submodules https://github.com/DongTienDinh/vi-asr-translate.git /kaggle/working/vi_asr_project
    %cd /kaggle/working/vi_asr_project
    !pip install -q -e .
else:
    %cd /kaggle/working/vi_asr_project

print("Ready for evaluation on device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# 1. Đánh giá ASR Checkpoints (B1, P1, P2) trên tập Test ViMD
!python -m vi_asr evaluate-asr \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml \
    --checkpoint runs/b1_ctc/checkpoint_best.pt \
    --split test \
    --run-dir eval/asr_b1

!python -m vi_asr evaluate-asr \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml \
    --checkpoint runs/p1_token_pooling/checkpoint_best.pt \
    --split test \
    --run-dir eval/asr_p1

!python -m vi_asr evaluate-asr \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml \
    --checkpoint runs/p2_token_temporal/checkpoint_best.pt \
    --split test \
    --run-dir eval/asr_p2

In [ ]:
# 2. Đánh giá Whisper-small Multilingual Zero-Shot Baseline (B_ref)
!python -m vi_asr baseline-whisper \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml \
    --model-size small \
    --split test

In [ ]:
# 3. Đánh giá Cascade ASR -> VinAI Translate (vi -> en)
!python -m vi_asr evaluate-cascade \
    --config configs/base.yaml \
    --paths configs/paths/kaggle.yaml \
    --checkpoint runs/p2_token_temporal/checkpoint_best.pt \
    --run-dir eval/cascade_p2

In [ ]:
# 4. Tổng hợp Bảng Kết quả So sánh và Kiểm định Bootstrap
!python -m vi_asr collect-results \
    --runs-root eval \
    --output artifacts/evaluation_summary.json